# Predicting the Sale Price of Bulldozers using Machine Learning

In this notebook, we're going to go through an example machine learning project with the goal of predicting the  predicting the sale price of bulldozers.

## 1. Problem defition

> How well can predict the future sale price of bulldozer,given its characteristics and previous example of how much similar bulldozers have been sold for?

## 2. Data

The data is download from the Kaggle Bluebook for Bulldozers competion: https://www.kaggle.com/c/bluebook-for-bulldozers/data

There are 3 main dataset

* Train.csv is the training set,which contains data through the end of 20111.

* Valid.csv is the validation set,which contains data from January 1,2012 - April 30,2012 You make predictions on this set throughout the majority of the competition. Your score on this set is used to create the public leaderboard.
* Test.csv is the test set,which won't be released untill the last week of the competition, It contains data from May 1, 2012 - November 2012. Your score on the set determines your final rank for the competition.

## 3. Evaluation

The evaluation metric for this competition is the RMSE (root mean squared log error) between the actual and predicted acution prices.

For more on the evaluation of this project check:
https://www.kaggle.com/c/bluebook-for-bulldozers/data

Note: The goal for most regression evaluation metrics is to minimize the error. For example, our goal for this project will be to build a machine learning model which minimizes RMSLE.

## 4. Features

Kaggle provides a data dictionary detailing all of the features of thr datasets. You can view this data dictionary on Google Sheets:


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [ ]:
# import training and validation sets
df = pd.read_csv("TrainAndValid.csv",low_memory=False)
df

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.columns

In [ ]:
fig,ax = plt.subplots()
ax.scatter(df['saledate'][:1000],df['SalePrice'][:1000])

In [ ]:
df.saledate[:1000]

In [ ]:
df.SalePrice.plot.hist()

In [ ]:
df.saledate.dtype

### Parsing dates
When we work with time series data,we want to enrich the time & date component as much as possible.

We can do that by telling pandas which of our columns has dates in it using the `parse_dates` parameter.

In [ ]:
# Import data again but this time parse dates
df = pd.read_csv("TrainAndValid.csv",
                 low_memory=False,
                 parse_dates=['saledate'])

In [ ]:
df.saledate.dtype

In [ ]:
df.saledate[:1000]

In [ ]:
fig,ax = plt.subplots()
ax.scatter(x=df['saledate'][:1000],y=df['SalePrice'][:1000])

In [ ]:
df.head()

In [ ]:
df.head().T

In [ ]:
df.saledate.head(20)

## **Sort DatFrame by saledate**
When working with time series data. It's a good idea to sort it by date.

In [ ]:
# Sort DataFrame in date order
df.sort_values(by=['saledate'],ascending=True,inplace=True)
df.saledate.head(20)


# Make a copy of the original DataFrame

We make a copy of the original dataframe so when we manipulate the copy,we've still got our original data.


In [ ]:
# Make a copy
df_temp = df.copy()

### Add datetime parameters for `saledate` column

In [ ]:
df_temp["saleYear"] = df_temp.saledate.dt.year
df_temp['saleMonth'] = df_temp.saledate.dt.month
df_temp['saleDay'] = df_temp.saledate.dt.day
df_temp['saleDayOfWeek'] = df_temp.saledate.dt.day_of_week
df_temp['saleDayOfYear'] = df_temp.saledate.dt.day_of_year

In [ ]:
df_temp.head().T

In [ ]:
# Now we've enriched our DataFrame with date time features,we can remove saledate
df_temp.drop("saledate",axis=1,inplace=True)


In [ ]:
df_temp.state.value_counts()

In [ ]:
df_temp.head()

In [ ]:
df_temp.info()

In [ ]:
df_temp['UsageBand'].dtype

In [ ]:
df_temp.isna().sum()

### Convert string to categories

One way we can turn all of our data into numbers is by converting them into pandas categories.

We can check the different datatypes go through the pandas.api.types documentation

In [ ]:
df_temp.head().T

In [ ]:
pd.api.types.is_string_dtype(df_temp['UsageBand'])

In [ ]:
# Find the columns which contain strings
for label,content in df_temp.items():
  if pd.api.types.is_string_dtype(content):
    print(label)

In [ ]:
for label, content in df_temp.items():
    if pd.api.types.is_object_dtype(content):
        print(label)

In [ ]:
# This will turn all of the string values intp category value
for label,content in df_temp.items():
  if pd.api.types.is_object_dtype(content):
    df_temp[label] = content.astype("category").cat.as_ordered()

In [ ]:
df_temp.info()

In [ ]:
df_temp.state.cat.categories

In [ ]:
df_temp.state.cat.codes

Thanks to pandas Categories we now have a way to access all of our data in the form of matters.

But we still have a bunch of missing data...

In [ ]:
# Check missing data
df_temp.isna().sum()/len(df_temp)

# Save preprocessed data

In [ ]:
# Export current temp datafrae
df_temp.to_csv("train_temp.csv",index=False)

In [ ]:
# Import preprocessed data
df_temp = pd.read_csv("train_temp.csv",low_memory=False)
df_temp.head().T

In [ ]:
df_temp.isna().sum()

## Fill missing values
### Fill numerical missing values first

In [ ]:
for label,content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    print(label)


In [ ]:
df_temp.ModelID

In [ ]:
# check for which numeric columns have null values
for label,content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      print(label)

In [ ]:
from pandas.core.dtypes.common import is_numeric_dtype
# Fill numeric rows with the median
for label,content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      # add a binary column which tells us if the data was missing or not
      df_temp[label+'_is_missing'] = pd.isnull(content)
      # fill missing numeric values with median
      df_temp[label] = content.fillna(content.median())

In [ ]:
# check if there's any null numeric values
for label,content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      print(label)

In [ ]:
df_temp.info()

In [ ]:
# check to see how many examples were missing
df_temp.auctioneerID_is_missing.value_counts()

# Filling and turning categorical variables into numbers

In [ ]:
# check for columns which aren't numeric
for label,content in df_temp.items():
  if not pd.api.types.is_numeric_dtype(content):
    print(label)


In [ ]:
# Turn categorical variables into numbers and fill missing
for label,content in df_temp.items():
  if not pd.api.types.is_numeric_dtype(content):
    # add binary column to indicate whether sample had missing value
    df_temp[label+"_is_missing"] = pd.isnull(content)
    # Turn categories into numbers and +1
    df_temp[label] = pd.Categorical(content).codes +1

In [ ]:
pd.Categorical(df_temp['state']).codes + 1

In [ ]:
df_temp.info()

In [ ]:
df_temp.head().T

In [ ]:
df_temp.isna().sum()

Now that all of data is numeric as well as our dataframe has no missing values,we should be abale to build a machine learning model.

5. Modelling

We've done enough EDA (we could always do more) but let's start to do some model-driven EDA.

In [ ]:
df_temp.head()

In [ ]:
%%time
# Instantiate model

# Let's build a machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_jobs=-1,
                              random_state=42)
model.fit(df_temp.drop("SalePrice",axis=1),df_temp['SalePrice'])



In [ ]:
# score the model
model.score(df_temp.drop("SalePrice",axis=1),df_temp['SalePrice'])


# Splitting data into train/validation sets

In [ ]:
df_temp.saleYear

In [ ]:
df_temp.saleYear.value_counts()

In [ ]:
 # Split data into training and validation
df_val = df_temp[df_temp.saleYear == 2012]
df_train = df_temp[df_temp.saleYear != 2012]
len(df_val),len(df_train)

In [ ]:
# Split data into x & y
x_train,y_train = df_train.drop("SalePrice",axis=1),df_train.SalePrice
x_valid,y_valid = df_val.drop("SalePrice",axis=1),df_val.SalePrice

x_train.shape,y_train.shape,x_valid.shape,y_valid.shape


# Building an evalutation function

In [ ]:
# Create evaluation function (the competition uses RMSLE)
from sklearn.metrics import r2_score,mean_squared_log_error,mean_absolute_error

def rmsle(y_test,y_preds):
  """
 Calculates root mean squared log error between predictions and true labels.
  """

  return np.sqrt(mean_squared_log_error(y_test,y_preds))

  # Create function to evaluate model on a few different levels

def show_scores(model):
  train_preds = model.predict(x_train)
  val_preds = model.predict(x_valid)
  scores = {"Training MSE":mean_absolute_error(y_train,train_preds),
            "Valid MSE":mean_absolute_error(y_valid,val_preds),
            "Training RMSLE":rmsle(y_train,train_preds),
            "Valid RMSLE":rmsle(y_valid,val_preds),
            "Training R^2":r2_score(y_train,train_preds),
            "Valid R^2":r2_score(y_valid,val_preds)}

  return scores

# Testing our model on asubset (to tune the hyperparameters)

In [ ]:
# Change max samples value
model = RandomForestRegressor(n_jobs=-1,random_state=42,max_samples=10000)


In [ ]:
%%time
# cutting down on the max number of sample each estimator can see improves training time
model.fit(x_train,y_train)

In [ ]:
show_scores(model)

# Hyperparameter tuning with RandomizedSearchCV

In [ ]:
model.get_params()

In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV

# Different RandomForestRegressor hyperparameters
rf_grid = {
    "n_estimators":np.arange(10,100,10),
    "max_depth":[None,3,5,10],
    "min_samples_split":np.arange(2,20,2),
    "min_samples_leaf":np.arange(1,20,2),
    "max_features":[0.5,1,"auto","sqrt"],
    "max_samples":[10000]
}

# rf = RandomizedSearchCV(model,random_state=42,n_jobs=-1)
rs_model = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1,random_state=42),
    param_distributions= rf_grid,
    n_iter=2,
    cv=5,
    verbose=True
)

# Fit the RandomizedSearchCV model
rs_model.fit(x_train,y_train)


In [ ]:
# Find the best model hyperparameters
rs_model.best_params_

In [ ]:
# Ealuate the RandomizedSearch model
show_scores(rs_model)

In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV

# Different RandomForestRegressor hyperparameters
rf_grid = {
    "n_estimators":np.arange(10,100,10),
    "max_depth":[None,3,5,10],
    "min_samples_split":np.arange(2,20,2),
    "min_samples_leaf":np.arange(1,20,2),
    "max_features":[0.5,1,"auto","sqrt"],
    "max_samples":[10000]
}

# rf = RandomizedSearchCV(model,random_state=42,n_jobs=-1)
rs_model = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1,random_state=42),
    param_distributions= rf_grid,
    n_iter=100,
    cv=5,
    verbose=True
)

# Fit the RandomizedSearchCV model
rs_model.fit(x_train,y_train)


In [ ]:
# Find the best model hyperparameters
rs_model.best_params_

In [ ]:
# Ealuate the RandomizedSearch model
show_scores(rs_model)

# Train a model with the best hyperparameters

**Note:** These were found after 100 iterations of `RandomizedSearchCV`.

In [ ]:
%%time

# Most ideal hyperparameters
ideal_model = RandomForestRegressor(n_estimators=40,
                                    min_samples_leaf=1,
                                    min_samples_split=14,
                                    max_features=0.5,
                                    n_jobs=-1,
                                    max_samples=None,
                                    random_state=42)

# Fit the ideal model
ideal_model.fit(x_train,y_train)

In [ ]:
# Scores for ideal model (trained on allthe data)
show_scores(ideal_model)

In [ ]:
# Scores on rs_model (only trained on - 10,000 examples)
show_scores(rs_model)

# Make predictions on test data

In [ ]:
# Import the test data
df_test = pd.read_csv("Test.csv")
df_test.head()

# Preprocessing the data (getting the test dataset in the same fromat as our training dataset)

In [ ]:
def preprocess_data(df):
  """
Performs transformations on dfand returns transformed df
  """
  df['saledate'] = pd.to_datetime(df['saledate'])

  df['saleYear'] = df.saledate.dt.year
  df['saleMonth'] = df.saledate.dt.month
  df['saleDay'] = df.saledate.dt.day
  df['saleDayOfWeek'] = df.saledate.dt.dayofweek
  df['saleDayOfYear'] = df.saledate.dt.dayofyear

  df.drop("saledate",axis=1,inplace=True)

  # Fill the numeric rows with median
  for label,content in df.items():
    if pd.api.types.is_numeric_dtype(content):
      if pd.isnull(content).sum():
        # Add a binary column which tells us if the data was missing or not
        df[label+'_is_missing'] = pd.isnull(content)
        # Fill missing numeric values with median
        df[label] = content.fillna(content.median())

  # Filled categorical missing data and turn categories into numbers
    if not pd.api.types.is_numeric_dtype(content):
      df[label+'_is_missing'] = pd.isnull(content)
      # we add +1 to the category code because pandas encodes missing categories as -1
      df[label] = pd.Categorical(content).codes+1

  return df

In [ ]:
# Process the test data
df_test = preprocess_data(df_test)
df_test.head()

In [ ]:
df_test.isna().sum().sort_values(ascending=False)

In [ ]:
# Make predictions on update test data
# test_data = ideal_model.predict(df_test)

In [ ]:
x_train.head()

In [ ]:
# We can find how the columns differ using sets
set(x_train.columns) - set(df_test.columns)


In [ ]:
# Manullay adjust df_test to have auctioneerID_is_missing column
df_test['auctioneerID_is_missing'] = False
df_test.head()

# Finally now our test dataframe has same features as our training dataframe,we can make predictions!

In [ ]:
# Make predictions on the test data
df_test = df_test[ideal_model.feature_names_in_]
test_preds = ideal_model.predict(df_test)

In [ ]:
test_preds

In [ ]:
# Format predictions into the same format Kaggle is after
df_preds = pd.DataFrame()
df_preds['SalesID'] = df_test['SalesID']
df_preds['SalesPrice'] = test_preds
df_preds

In [ ]:
# Export prediction data
df_preds.to_csv("test_prediction.csv",index=False)

**Feature Importance**

Feature importance seeks to figure out which different attributes of the data were most importances when it comes to predicting the **target variables** (SalePrice).

In [ ]:
# Find feature importance of our best model
len(ideal_model.feature_importances_)

In [ ]:
x_train

In [ ]:
# Helper function for plotting feature importance
def plot_features(columns,importances,n=20):
  df = (pd.DataFrame({"features":columns,
                      "feature_importances":importances})
        .sort_values("feature_importances",ascending=False)
        .reset_index(drop=True))

  # Plot the dataframe
  fig,ax = plt.subplots()
  ax.barh(df['features'][:n],df['feature_importances'][:20])
  ax.set_ylabel("Features")
  ax.set_xlabel("Feature importance")
  ax.invert_yaxis()

In [ ]:
plot_features(x_train.columns,ideal_model.feature_importances_)

In [ ]:
df['ProductSize'].value_counts()